# 02 · Três métodos de seleção e votação das features

Regra fixada **antes** da comparação (`config.yaml → votacao`): cada método ranqueia as features candidatas
e vota nas suas **k = 10** melhores. Uma feature é eleita com **≥ 2 dos 3 votos**. Se sobrarem menos de **6**,
completa-se pela menor média de ranking, que também desempata e ordena.

Os três métodos são ajustados **só com o treino do ciclo**, numa amostra estratificada de 200 mil linhas:

| Método | O que mede |
|---|---|
| `SelectKBest(mutual_info_classif)` | dependência estatística (inclusive não linear) entre cada feature e o alvo |
| Importância de `RandomForestClassifier` | quanto cada feature reduz a impureza nas árvores |
| `RFE(LogisticRegression)` | ordem em que a feature é eliminada por ter o menor coeficiente |

In [1]:
import sys, json, pathlib, warnings
RAIZ = pathlib.Path.cwd().resolve()
while not (RAIZ / "config" / "config.yaml").exists():  # funciona rodando de notebooks/ ou da raiz
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", message=".*Hint: Inferred schema.*")

import pandas as pd
from src.comum import carregar_config, caminho, estado, ler_ciclo, atualizar_ciclo, configurar_mlflow, tags_ciclo, pasta_evidencias, gatilho
cfg = carregar_config()
pd.set_option("display.width", 160)

In [2]:
from src import features as fs, preprocessamento as pp
ciclo = ler_ciclo(cfg)
treino = pd.read_parquet(estado(cfg, "processados") / "treino.parquet")
v = cfg["votacao"]
amostra = fs.amostrar(treino, v["amostra"], v["seed"])
print(ciclo["cycle_id"], ciclo["data_version"], "| amostra:", len(amostra), "linhas,",
      f"{amostra[pp.ALVO].mean():.2%} positivos | candidatas:", len(pp.CANDIDATAS))

20261128-1731-46170804405 v2024-05-ef7d78aa | amostra: 200000 linhas, 2.09% positivos | candidatas: 20


In [3]:
tabelas = {nome: metodo(amostra, pp.CANDIDATAS, v["k"], v["seed"]) for nome, metodo in fs.METODOS.items()}
pd.concat({n: t[["score", "ranking", "voto"]] for n, t in tabelas.items()}, axis=1).sort_values(("random_forest", "ranking")).round(4)

/opt/hostedtoolcache/Python/3.12.16/x64/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


selectkbest_mi              random_forest              rfe_logistica             
                                score ranking voto         score ranking voto         score ranking voto
feature                                                                                                 
taxa_historica_periodo         0.0155       1    1        0.2401       1    1            20       1    1
taxa_historica                 0.0147       5    1        0.1935       2    1             5      16    0
acid_ult_365d                  0.0144       6    1        0.1426       3    1            16       5    1
vmd                            0.0150       3    1        0.0966       4    1             7      14    0
lon                            0.0149       4    1        0.0660       5    1             8      13    0
delegacia                      0.0074       7    1        0.0414       6    1            19       2    1
temperatura                    0.0004      12    0        0.0328       7    1            12       9    1
nebulosidade                   0.0016      10    1        0.0266       8    1             2      19    0
periodo_idx                    0.0009      11    0        0.0261       9    1            17       4    1
br                             0.0060       8    1        0.0227      10    1            18       3    1
trecho_km                      0.0032       9    1        0.0224      11    0             9      12    0
lat                            0.0152       2    1        0.0207      12    0            14       7    1
chuva_mm                       0.0001      16    0        0.0195      13    0            13       8    1
dia_semana                     0.0002      13    0        0.0191      14    0            15       6    1
mes                            0.0000      17    0        0.0145      15    0             6      15    0
fim_de_semana                  0.0001      14    0        0.0082      16    0            11      10    1
pandemia                       0.0001      15    0        0.0034      17    0            10      11    0
temporada_verao                0.0000      18    0        0.0023      18    0             4      17    0
vespera_ou_pos_feriado         0.0000      19    0        0.0009      19    0             3      18    0
feriado                        0.0000      20    0        0.0004      20    0             1      20    0

## Tabela de votos e features eleitas

In [4]:
votacao = fs.votar(tabelas, v["min_votos"], v["min_features"])
eleitas = list(votacao.index[votacao["eleita"]])
print(f"{len(eleitas)} features eleitas:", eleitas)
votacao

11 features eleitas: ['taxa_historica_periodo', 'acid_ult_365d', 'delegacia', 'br', 'lat', 'vmd', 'lon', 'taxa_historica', 'periodo_idx', 'temperatura', 'nebulosidade']


,voto_selectkbest_mi,voto_random_forest,voto_rfe_logistica,rank_selectkbest_mi,rank_random_forest,rank_rfe_logistica,votos,media_ranking,eleita,motivo
feature,,,,,,,,,,
taxa_historica_periodo,1,1,1,1,1,1,3,1.00,True,≥ 2 votos
acid_ult_365d,1,1,1,6,3,5,3,4.67,True,≥ 2 votos
delegacia,1,1,1,7,6,2,3,5.00,True,≥ 2 votos
br,1,1,1,8,10,3,3,7.00,True,≥ 2 votos
lat,1,0,1,2,12,7,2,7.00,True,≥ 2 votos
vmd,1,1,0,3,4,14,2,7.00,True,≥ 2 votos
lon,1,1,0,4,5,13,2,7.33,True,≥ 2 votos
taxa_historica,1,1,0,5,2,16,2,7.67,True,≥ 2 votos
periodo_idx,0,1,1,11,9,4,2,8.00,True,≥ 2 votos


In [5]:
ev = pasta_evidencias(cfg, ciclo)
votacao.to_csv(ev / "votacao.csv")
mlflow = configurar_mlflow(cfg)
with mlflow.start_run(run_name=f"{ciclo['cycle_id']} · votação", tags=tags_ciclo(ciclo, etapa="votacao")) as run:
    mlflow.log_params({"metodos": ",".join(fs.METODOS), "k": v["k"], "min_votos": v["min_votos"],
                       "min_features": v["min_features"], "desempate": v["desempate"], "amostra": len(amostra),
                       "seed": v["seed"], "features_eleitas": ",".join(eleitas)})
    mlflow.log_metric("n_features_eleitas", len(eleitas))
    mlflow.log_artifact(ev / "votacao.csv")
    for nome, t in tabelas.items():
        t.to_csv(ev / f"ranking_{nome}.csv"); mlflow.log_artifact(ev / f"ranking_{nome}.csv", "rankings")
ciclo = atualizar_ciclo(cfg, features_eleitas=eleitas, run_votacao=run.info.run_id)